# TriviaQA experiments

A working notebook for one TriviaQA cell (dataset × model under test). Everything
TriviaQA is organised here in the order a run happens:

| § | Step | API calls? |
|---|---|---|
| 1 | Setup: pick the config | no |
| 2 | Raw data: what TriviaQA looks like, duplicates, answers | no |
| 3 | Budget: calls this cell will cost | no |
| 4 | Results folders under `experiments/results/` | no |
| 5 | **Collection**: runs only when `SPEND = True` | **yes** |
| 6 | Collection health: extraction failures, agreement, the stable-but-wrong regime | no |
| 7 | Dev-slice audit: freeze prompts/thresholds here, never on eval items | no |
| 8 | Evaluation: ladder + ablations, confirmatory test, budget curve, signal effects (LOO + Shapley), Prop. 3, robustness | no |
| 9 | Drill-down: one case, its draws, probes and evidence ledger | no |
| 10 | Paper figures and tables for this run | no |

**Rules** (CLAUDE.md): collection is the only API spend; nothing is tuned on eval
items; results from `configs/simulation.yaml` are pipeline checks, never evidence;
the smoke run (1 batch) cannot power the confirmatory test. The logic lives in
`confledger` and `data`. This notebook only calls it and
displays the output, so keep new logic out of it.

## How to run the smoke pipeline

The smoke run is TriviaQA × gpt-4.1 on 60 questions (10 dev + 50 eval, one batch of
50); §3 shows its cost. Evaluation (§8 onward) needs `n_paraphrases` and `n_same` of at
least 12; a smaller collection tests the collection plumbing only. Every step can run from a terminal (repo root) or from
this notebook; both call the same `confledger` functions and write to
`experiments/results/smoke/`.

**Before the first run**
1. `pip install -e ".[openai,data,notebook]"`
2. Fill the generator and judge model ids in `configs/smoke.yaml` (OpenAI models,
   not gpt-4.1). Collection refuses to start while any id is a `<placeholder>`.
3. Transport: leave `LLM_BACKEND` unset and export `OPENAI_API_KEY` (direct API),
   or `export LLM_BACKEND=safechain` in the private environment.

| Step | Terminal | Here | API calls |
|---|---|---|---|
| Download + check data | `python -m data.prepare -c configs/smoke.yaml` | §2 | 0 |
| Estimate cost | `python experiments/scripts/estimate.py -c configs/smoke.yaml --reseed-batches 0` | §3 | 0 |
| **Collect** | `python experiments/scripts/collect.py -c configs/smoke.yaml --reseed-batches 0 -v` | §5 with `SPEND = True` | **see §3** |
| Check the collection | | §6, §7 | 0 |
| Evaluate (Sec. 4 protocol) | `python experiments/scripts/evaluate.py -c configs/smoke.yaml` | §8 | 0 |
| Paper figures + tables | `python experiments/scripts/<script>.py -c configs/smoke.yaml` | §10 | 0 |

`experiments/scripts/run_all.sh configs/smoke.yaml` runs every terminal step in order
(it collects with 2 re-drawn batches by default; for the smoke run use the table's
commands). To rehearse without spending, set `CONFIG` to a simulated config: every
section then runs for free, and its numbers are pipeline checks, not evidence.

## 1. Setup

In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
os.chdir(ROOT)                      # config paths are relative to the repo root

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

from confledger.collection import collection_path, estimate_calls, load_collection
from confledger.utility.config import load_config, model_spec
from confledger.evaluation_protocol import cell_results
from data import load_split
from confledger.evaluation_protocol import evaluate
from confledger.methods import ledger_statement, cel, registry
from confledger.sprt_posterior import majority

pd.set_option("display.max_colwidth", 120)

In [ ]:
# ---- parameters -------------------------------------------------------------
# One config = one run; outputs go to experiments/results/<config name>/.
CONFIG = os.environ.get("CONFLEDGER_CONFIG", "configs/smoke.yaml")
DATASET = "triviaqa"
MODEL = None            # model id from `models:` (e.g. "gpt-4.1"); None = the first one
SPEND = False           # True allows API calls in §5. Leave False until the run is approved.
RESEED_BATCHES = 0      # re-drawn batches for the Jaccard stability check (0 for the smoke run)
FORCE = False           # recompute protocol results from the collections

cfg = load_config(CONFIG, [DATASET], [MODEL] if MODEL else None)
MODEL = cfg["_models"][0]
SYNTHETIC = bool(cfg.get("synthetic"))
TAG = f"{DATASET}__{MODEL}"
RESULTS = Path(cfg["paths"]["results"])
display(Markdown(f"""
| | |
|---|---|
| config | `{CONFIG}` {"(**simulated: pipeline check, not evidence**)" if SYNTHETIC else ""} |
| model under test | `{model_spec(cfg, MODEL)}` |
| generator | `{cfg["generator"]}` |
| judge | `{cfg["judge"]}` |
| transport | `LLM_BACKEND={os.environ.get("LLM_BACKEND", "") or "unset (direct API)"}` |
| split | {cfg["split"]}, k = {cfg["k"]} |
| outputs | collections `{cfg["paths"]["collections"]}`, results `{RESULTS}` |
"""))

## 2. Raw data (no API calls)

TriviaQA `rc.nocontext` validation, turned into verification items: *"Is
[candidate] the correct answer?"*, half gold and half a distractor written by the
generator model. The loader removes duplicate `question_id`s first, because most
questions appear twice (web + wiki), see decisions.md #7. Distractors need the
generator, so the actual items only exist after collection starts (§5).

In [ ]:
if SYNTHETIC:
    print("synthetic config: no raw data")
else:
    from datasets import load_dataset
    from data.loaders import HF_SOURCES
    hf_id, hf_cfg, hf_split = HF_SOURCES[DATASET]
    raw = load_dataset(hf_id, hf_cfg, split=hf_split)
    qids = raw["question_id"]
    print(f"{hf_id} [{hf_cfg}] {hf_split}: {len(raw):,} rows, {len(set(qids)):,} unique questions; "
          f"need {cfg['split']['n_dev'] + cfg['split']['n_eval']}")
    sample = raw.select(range(0, 2000, 250))
    display(pd.DataFrame({"question": sample["question"],
                          "gold": [a["value"] for a in sample["answer"]],
                          "n_aliases": [len(a["aliases"]) for a in sample["answer"]]}))

## 3. Budget (no API calls)

In [ ]:
est = pd.DataFrame(estimate_calls(cfg, RESEED_BATCHES))
est["total"] = est[["target", "judge", "generator"]].sum(axis=1)
TOTAL_CALLS = int(est["total"].sum())
est

## 4. Results folders

One config in `configs/` = one run = one folder in `experiments/results/`. For a
variant, copy the config under a new name (and its `paths`). Don't edit a config
that already has results.

In [ ]:
runs = sorted(p for p in Path("experiments/results").glob("*/collections"))
if runs:
    display(pd.DataFrame([{"run": p.parent.name,
                           "collected": sorted(f.name for f in p.glob("*.jsonl")),
                           "evaluated": sorted(f.name for f in (p.parent / "results").glob("*.json"))}
                          for p in runs]))
else:
    print("nothing collected yet")

## 5. Collection (**spends API calls**)

Runs only with `SPEND = True`, or on a simulated config, which is free. Collection
is resumable: finished items are skipped and cached responses cost nothing. Every
collect/evaluate appends seed, environment and git commit to `<results>/run_log.jsonl`.
`check_config` refuses to start while any model id is still a `<placeholder>`.

In [ ]:
if SPEND or SYNTHETIC:
    from confledger.collection import collect_config
    collect_config(cfg, RESEED_BATCHES)
else:
    print(f"SPEND=False: collection skipped. Would cost ~{TOTAL_CALLS:,} calls.")

## 6. Collection health (no API calls)

One row per eval item: the majority over the 12 fresh-paraphrase draws, its
agreement, and the four probes. Rows that are **unanimous but wrong** are the
stable-but-wrong regime TriviaQA was chosen for (memorised wrong facts). Resampling
cannot catch them; only the invariance probes can.

In [ ]:
def case_table(coll: dict) -> pd.DataFrame:
    rows = []
    for cd in coll.values():
        ans, m, n = majority([d.answer for d in cd.fresh])
        rows.append({"item_id": cd.item_id, "gold": cd.gold, "answer": ans, "agree": f"{m}/{n}",
                     "unanimous": ans is not None and m == n, "correct": ans == cd.gold,
                     "parse_fail": sum(d.answer is None for d in cd.fresh + cd.same),
                     **{f"probe_{g}": a for g, a in cd.probes.items()},
                     "verbal_conf": cd.verbal_conf,
                     "n_clusters": len({d.cluster for d in cd.fresh if d.cluster is not None})})
    return pd.DataFrame(rows).set_index("item_id")


coll_path = collection_path(cfg, DATASET, MODEL)
assert coll_path.exists(), "no collection yet: run §5 first"
coll = load_collection(coll_path)
tab = case_table(coll)
n_draws = sum(len(cd.fresh) + len(cd.same) for cd in coll.values())
display(Markdown(f"**{len(tab)} eval items** | majority accuracy {tab.correct.mean():.3f} | "
                 f"extraction failures {tab.parse_fail.sum()}/{n_draws} draws | "
                 f"gold yes/no {dict(tab.gold.value_counts())}"))
display(pd.crosstab(tab.unanimous, tab.correct, margins=True)
        .rename_axis(index="unanimous 12/12", columns="correct"))
tab.head(10)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
tab.groupby("agree", sort=False).correct.agg(["mean", "size"]).sort_index()["mean"].plot.bar(
    ax=ax[0], title="accuracy by fresh-draw agreement")
for ok, g in tab.groupby("correct"):
    g.verbal_conf.plot.hist(ax=ax[1], alpha=0.5, bins=10, label=f"correct={ok}")
ax[1].set_title("verbalized confidence (RLHF pile-up expected)")
ax[1].legend()
fig.tight_layout()

## 7. Dev-slice audit (freeze here, never on eval items)

Gates before the full grid: answer extraction ≥ 0.98 and paraphrase meaning
preservation ≥ 0.95. Paraphrases live in the generator cache. The audit script
exports them for hand-labelling:

```bash
python experiments/scripts/dev_audit.py -c <CONFIG> -m <MODEL> export   # then fill the CSVs
python experiments/scripts/dev_audit.py -c <CONFIG> -m <MODEL> score
```

Under SafeChain, also check that redaction left the question text intact
(decisions.md #6).

In [ ]:
dev_path = collection_path(cfg, DATASET, MODEL, "__dev")
if dev_path.exists():
    dev = load_collection(dev_path)
    dtab = case_table(dev)
    print(f"dev slice: {len(dtab)} items, extraction failures {dtab.parse_fail.sum()}")
    cd = next(iter(dev.values()))
    display(pd.DataFrame([{"answer": d.answer, "cluster": d.cluster, "explanation": d.explanation}
                          for d in cd.fresh]).head(6))
else:
    print("no dev collection yet")

## 8. Evaluation (no API calls)

The full Sec. 4 protocol, cached in `<results>/<TAG>.json`. Set `FORCE = True` to
recompute it after re-collecting. Baselines are matched to the pipeline's realised
calls per case.

In [ ]:
res = cell_results(cfg, DATASET, MODEL, force=FORCE)
if res["n_batches"] < 5:
    display(Markdown(f"**{res['n_batches']} batches: the confirmatory test is underpowered; "
                     "treat this as a pipeline check.**"))
ladder = pd.DataFrame({n: {"prec@k": m["mean_prec"], "calls": m["mean_calls"], "AUROC": m["auroc"],
                           "PRR": m["prr"], "survival": m.get("survival")}
                       for n, m in res["methods"].items()}).T
display(Markdown(f"matched budget: **{res['matched_budget']} calls/case**"))
ladder.style.format("{:.3f}")

In [ ]:
c = res["confirmatory"]
display(Markdown(f"**Confirmatory** (CEL − self-consistency, paired over batches): "
                 f"Δ = {c['mean_diff']:+.3f} [{c['ci_lo']:+.3f}, {c['ci_hi']:+.3f}], "
                 f"wins {c['wins']}/{c['n']}, Wilcoxon p = {c['wilcoxon_p']:.3f}"))
display(Image(RESULTS / f"{TAG}_budget.png"), Image(RESULTS / f"{TAG}_strata.png"))

### Budget-allocated CEL-k vs self-consistency with k draws

The deployment comparison. For each budget k, CEL-k spreads its calls over a **screen**
(k₁ draws; any disagreement eliminates) and, for survivors only, extra draws, flips,
challenge and SE (1 judge call); verbal confidence rides on every draw. Among ~1,200
allocations, the one with the best dev score whose **expected** calls per case on the dev
slice are ≤ k is frozen, then scored once on the eval items. CEL-k is compared on its
expected calls (batch average), self-consistency on exactly k draws. No certificate.

In [ ]:
bm = res["budget_matched"]
display(Markdown(f"allocation chosen on **{bm.get('n_dev', 0)} dev items** by {bm.get('metric', bm.get('note'))}"))
pd.DataFrame({k: {"CEL-k allocation": b["allocation"],
                  "E[calls] dev": b["dev_calls"], "E[calls] eval": b["cel_k"]["calls"],
                  "CEL-k prec@10": b["cel_k"]["prec"],
                  "SC-k prec@10": b["self_consistency"]["prec"],
                  "SC-k+verbal": b["self_consistency_verbal"]["prec"], "Δ vs SC": b["paired"]["mean_diff"],
                  "Δ vs SC+verbal": b["paired_vs_verbal"]["mean_diff"],
                  "95% CI": f"[{b['paired']['ci_lo']:+.3f}, {b['paired']['ci_hi']:+.3f}]",
                  "wins": f"{b['paired']['wins']}/{b['paired']['n']}"}
              for k, b in bm["by_k"].items()}).T.rename_axis("k (calls/case)")

### Signal effects on confidence estimation

How much each of the five signals (lower bound *L*, flip/invariance probes, rationale
entropy SE, challenge, verbal confidence) contributes, measured two ways:

- **Leave-one-out (LOO):** CEL minus CEL with that signal removed. Δprec@10 is paired over
  batches (95% CI); ΔAUROC and ΔPRR judge the whole ranking; *calls saved* is what
  dropping the signal would save per case.
- **Shapley φ:** the signal's fair share of the prec@10 gain over random-among-survivors,
  averaged over all 32 signal combinations (Borda ranking, screen fixed). It credits
  signals that are redundant with each other, which LOO cannot.

A signal with LOO ≈ 0 but φ > 0 is useful yet covered by another signal. With one
batch (the smoke run) the CIs are uninformative.

In [ ]:
sh, loo = res["shapley"], res["leave_one_out"]
effects = pd.DataFrame({
    j: {"LOO Δprec@10": loo[j]["d_prec"], "LOO CI": f"[{loo[j]['ci'][0]:+.3f}, {loo[j]['ci'][1]:+.3f}]",
        "LOO ΔAUROC": loo[j]["d_auroc"], "LOO ΔPRR": loo[j]["d_prr"], "calls saved": loo[j]["calls_saved"],
        "φ prec@10": sh["phi"][j], "φ CI": f"[{sh['phi_ci'][j][0]:+.3f}, {sh['phi_ci'][j][1]:+.3f}]"}
    for j in sh["phi"]}).T.sort_values("φ prec@10", ascending=False)
display(effects.style.format({c: "{:+.3f}" for c in effects.columns if "CI" not in c and c != "calls saved"}
                             | {"calls saved": "{:.2f}"}))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
effects["LOO Δprec@10"].astype(float).plot.barh(ax=ax[0], title="leave-one-out Δprec@10")
effects["φ prec@10"].astype(float).plot.barh(ax=ax[1], title="Shapley φ (prec@10)")
for a in ax:
    a.axvline(0, color="gray", lw=.8)
fig.tight_layout()

In [ ]:
# Prop. 3 (MODEL-BASED): evidence arms and exchange rates; robustness
arms = pd.DataFrame(res["arms"]).T[["mean_prec", "mean_calls"]]
kap = pd.DataFrame({"kappa derived": res["kappa_derived"], "kappa_hat": res["kappa_hat"]["kappa_hat"]})
display(arms, kap)
display(pd.Series({"screen gain": sh["screen_gain"], "flip increment": sh["flip_increment"],
                   **{f"held-out {k}": v for k, v in res["heldout_audit"].items()},
                   **{f"jaccard {k}": v for k, v in res["jaccard"].items() if k != "per_batch"}},
                  dtype=object).to_frame("robustness"))

## 9. Drill-down: one case

The pipeline's ranking of the first eval batch, then one case in full: its draws,
probes, and the evidence ledger (nats; certification needs the balance ≥ A).
By default it picks a **selected but wrong** case, the failure worth reading
first. Set `ITEM` to inspect a specific one.

In [ ]:
items, split = load_split(cfg, DATASET)   # distractors come from the generator cache
by_id = {it.item_id: it for it in items}
o = evaluate(coll, split.batches, registry(res["matched_budget"])["cel"], cfg["k"])
cases = {c.item_id: c for c in o["_cases"]}
batch0 = pd.DataFrame([{"rank": o["_ranks"][i], "item_id": i, "selected": i in o["selected"][0],
                        "survived": cases[i].survived, "answer": cases[i].answer,
                        "correct": cases[i].correct,
                        **{k: round(v, 3) for k, v in cases[i].signals.items() if k in ("L", "flip", "se", "llr")}}
                       for i in split.batches[0]]).sort_values("rank").set_index("rank")
batch0.head(15)

In [ ]:
ITEM = None
wrong = [i for b in o["selected"] for i in b if not cases[i].correct]
ITEM = ITEM or (wrong[0] if wrong else o["selected"][0][0])
cd, it = coll[ITEM], by_id[ITEM]
st = ledger_statement(cd)
display(Markdown(f"### `{ITEM}`\n\n```\n{it.question}\n```\n\ngold **{cd.gold}**, pipeline answer "
                 f"**{st['answer']}**, survived screen: {st['survived']}, "
                 f"balance **{st['balance_nats']:+.2f}** nats (k_eff {st['k_eff']:.1f}), "
                 f"certified: {st['certified']}, calls {st['calls']}"))
display(pd.DataFrame(st["entries"], columns=["entry", "nats"]).assign(running=lambda d: d.nats.cumsum()))
display(pd.DataFrame([{"answer": d.answer, "cluster": d.cluster, "explanation": d.explanation}
                      for d in cd.fresh]))
pd.Series({**cd.probes, "challenge": f"{cd.challenge_base} -> {cd.challenge_answer}",
           "verbal": f"{cd.verbal_answer} @ {cd.verbal_conf}"}).to_frame("probes / other")

## 10. Paper figures and tables (no API calls)

Runs the seven paper scripts on this config's evaluated cell, exactly as the terminal
command `python experiments/scripts/<script>.py -c CONFIG` would, then shows the
figures. Outputs go to `<results>/paper/`: PDFs and `.tex` tables for the paper, and
a PNG copy of each figure for this notebook. Needs §8 (the evaluated cell) first.
With one batch, the confirmatory and paired tests in these tables are not evidence.

In [ ]:
import subprocess
import sys

PAPER_SCRIPTS = {                      # script -> what it produces for the paper
    "sprt_characteristics": "Sec. 3 constants; truncated-SPRT characteristics; draws vs guarantee (sprt_settings.tex)",
    "main_ladder": "Table: baseline ladder + ablations; confirmatory test (Holm)",
    "budget_curves": "Headline figure (CEL, CEL-k, self-consistency vs calls); budget-matched table; horizontal gap",
    "evidence_exchange": "Prop. 3: evidence arms A/B/C; kappa vs kappa-hat",
    "monotonicity": "Assumption 1: accuracy by rank stratum",
    "signal_effects": "Signal effects: leave-one-out + Shapley on prec@10",
    "robustness": "Selection stability, held-out audit, judge swap",
}
assert (RESULTS / f"{TAG}.json").exists(), "no evaluated cell yet: run §8 first"
for name, what in PAPER_SCRIPTS.items():
    r = subprocess.run([sys.executable, f"experiments/scripts/{name}.py", "-c", CONFIG,
                        "-d", DATASET, "-m", MODEL], capture_output=True, text=True)
    print(f"{'ok    ' if r.returncode == 0 else 'FAILED'} {name}.py  ({what})")
    if r.returncode:
        print(r.stderr[-1500:])
PAPER = RESULTS / "paper"

In [ ]:
for png, title in [(f"budget_curves_{MODEL}.png", "Headline: quality vs calls per case"),
                   ("monotonicity.png", "Assumption 1: accuracy by rank stratum"),
                   ("truncation_oc.png", "Truncated SPRT: P(certify) and P(eliminated) vs stability")]:
    display(Markdown(f"**{title}** (`{png}`)"), Image(PAPER / png, width=720))

In [ ]:
tex = sorted(PAPER.glob("*.tex"))
display(Markdown("| LaTeX table | size |\n|---|---|\n" +
                 "\n".join(f"| `{p.name}` | {p.stat().st_size:,} B |" for p in tex)))
print((PAPER / "main_ladder.tex").read_text())     # e.g. the main table, ready for \input{}

## Notes

Record observations here, under a dated heading per run.